# ARC-AGI Solver — Evaluate

Runs inference on a trained checkpoint stored in Google Drive.
Evaluation is leave-one-out over the original ARC training pairs for each task.

**Cell order:**
1. Check GPU → 2. Mount Drive → 3. Clone repo + ARC data → 4. Config → 5. Evaluate

**Modes:**
- `greedy`        — single forward pass, argmax per cell
- `tta`           — N colour permutations → un-permute → majority vote per cell
- `ttt`           — fine-tune on task training pairs (M steps) then run TTA
- `all`           — run greedy / TTA / TTT independently and print a comparison table
- `selective_ttt` — **recommended**: TTA on all tasks; TTT only on tasks TTA doesn't
                    get perfect; takes best of TTA vs TTT per task. Protects already-
                    perfect tasks from TTT regression. ~60% faster than full TTT.

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — inference will be slow on CPU.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints'

print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')
ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pt'))
print(f'Available checkpoints ({len(ckpts)}):')
for p in ckpts:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, urllib.request, zipfile
from pathlib import Path

# Clone / pull repo
REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
if not os.path.isdir(REPO):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git')
else:
    os.system(f'git -C {REPO} pull --ff-only')
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')
os.system('git log --oneline -3')

# ARC training data (evaluate.py reads from data/training/)
DATA_DIR = Path('data')
if (DATA_DIR / 'training').exists() and len(list((DATA_DIR / 'training').glob('*.json'))) >= 400:
    print(f"ARC data already present ({len(list((DATA_DIR/'training').glob('*.json')))} tasks).")
else:
    print('Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list((DATA_DIR/'training').glob('*.json')))} tasks")

In [ ]:
# ── Cell 4: Evaluation configuration ────────────────────────────────────────
from pathlib import Path

RUN_NAME  = 'all_400_arc'
K_CONTEXT = 3
ANALYZE            = True
COMPARE_RULE_BASED = True

# ── Inference options ─────────────────────────────────────────────────────────
EVAL_MODE  = 'selective_ttt'  # 'greedy' | 'tta' | 'ttt' | 'all' | 'selective_ttt'
N_PERMS    = 10               # colour permutations for TTA / TTT
N_D4       = 1                # D4 orientations: 1=colour-only, 8=all symmetries

# ── TTT settings ─────────────────────────────────────────────────────────────
# TTT_STEPS: maximum steps — early stopping will cut this short if LOO peaks.
# TTT_EVAL_EVERY: check LOO every N steps (was 20; 5 means save-best actually fires).
# TTT_PATIENCE: stop if no improvement for this many eval cycles (5 cycles × 5 steps = 25 steps of no progress).
TTT_STEPS      = 100    # max steps; early stopping takes over
TTT_LR         = 1e-4
TTT_EVAL_EVERY = 5      # must be << TTT_STEPS so save-best triggers
TTT_PATIENCE   = 5      # early-stop after 5 eval cycles with no improvement

# ── Pilot mode: restrict to specific task IDs ─────────────────────────────────
# Set to a list of task IDs to run on just those tasks (useful for hyperparameter
# search). Set to None to run on all tasks from the checkpoint.
PILOT_TASK_IDS = [
    '007bbfb7', '00d62c1b', '045e512c', '06df4c85', '09629e4f',
    '0b148d64', '0d3d703e', '0dfd9992', '10fcaaa3', '137eaa0f',
    '1b2d62fb', '1c786137', '1cf80156', '239be575', '253bf280',
]
# PILOT_TASK_IDS = None  # ← uncomment to run all tasks

# ── Resolve checkpoint path ───────────────────────────────────────────────────
ckpt_path = str(Path(DRIVE_CKPT_DIR) / f'transformer_c{RUN_NAME}_best.pt')
if Path(ckpt_path).exists():
    sz = Path(ckpt_path).stat().st_size / 1e6
    print(f'Checkpoint: {ckpt_path}  ({sz:.1f} MB)  ✓')
else:
    print(f'Checkpoint NOT FOUND: {ckpt_path}')
    print('Available:')
    for p in sorted(Path(DRIVE_CKPT_DIR).glob('*.pt')):
        print(f'  {p.name}')

n_tasks = len(PILOT_TASK_IDS) if PILOT_TASK_IDS else 'all'
print(f'Tasks: {n_tasks}  |  TTT: steps={TTT_STEPS}  eval_every={TTT_EVAL_EVERY}  patience={TTT_PATIENCE}  lr={TTT_LR}')


In [ ]:
# ── Cell 5: Run evaluation ───────────────────────────────────────────────────
# Pilot (15 tasks, selective_ttt, steps=100, early stopping):  ~20-30 min
# Full  (400 tasks, selective_ttt, steps=100, early stopping): ~2-4 hrs
import subprocess, sys, shutil
from pathlib import Path

# ── Restore TTA cache from Drive (so Phase 1 is skipped if already run) ───
drive_results = Path(DRIVE_BASE) / 'results'
local_results = Path('results')
local_results.mkdir(exist_ok=True)
restored = []
if drive_results.exists():
    for cached in sorted(drive_results.glob('tta_cache_*.json')):
        dest = local_results / cached.name
        if not dest.exists():
            shutil.copy(cached, dest)
            restored.append(cached.name)
if restored:
    print(f'Restored TTA cache from Drive: {restored}')
else:
    print('No TTA cache found on Drive — Phase 1 will run in full.')

cmd = [
    sys.executable, '-u', 'scripts/evaluate.py',
    '--checkpoint',     ckpt_path,
    '--mode',           EVAL_MODE,
    '--n-perms',        str(N_PERMS),
    '--n-d4',           str(N_D4),
    '--ttt-steps',      str(TTT_STEPS),
    '--ttt-lr',         str(TTT_LR),
    '--ttt-eval-every', str(TTT_EVAL_EVERY),
    '--ttt-patience',   str(TTT_PATIENCE),
    '--k-context',      str(K_CONTEXT),
    '--verbose',
    *([ '--analyze']            if ANALYZE            else []),
    *([ '--compare-rule-based'] if COMPARE_RULE_BASED else []),
    *([ '--task-ids'] + PILOT_TASK_IDS if PILOT_TASK_IDS else []),
]

print(f'Mode: {EVAL_MODE}  |  tasks={n_tasks}  n_perms={N_PERMS}  '
      f'ttt_steps={TTT_STEPS}  eval_every={TTT_EVAL_EVERY}  patience={TTT_PATIENCE}  lr={TTT_LR}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

# ── Save TTA cache + analysis JSON to Drive ────────────────────────────────
drive_results.mkdir(parents=True, exist_ok=True)
for cached in sorted(local_results.glob('tta_cache_*.json')):
    shutil.copy(cached, drive_results / cached.name)
    print(f'TTA cache saved to Drive: {cached.name}')
if ANALYZE and proc.returncode == 0:
    import glob as _glob
    for j in sorted(_glob.glob('results/error_analysis_*.json')):
        shutil.copy(j, drive_results / Path(j).name)
        print(f'Analysis saved to Drive: {Path(j).name}')
